### ライブラリインポート

In [112]:
import yaml
import requests
import pandas as pd
from joblib import Parallel, delayed
from pathlib import Path
import zipfile

from datetime import date, datetime, timedelta

### パラメータ読み込み

In [75]:
config_path = "../input/conf/"

with open(config_path + "credentials.yml", "r") as f:
    credentials = yaml.safe_load(f)

with open(config_path + "parameters.yml", "r") as f:
    params = yaml.safe_load(f)

EDINET_API_KEY = credentials["edinet"]["key"]

edinet_base_url = params["edinet"]["base_url"]
Path_docs_list = params["edinet"]["paths"]["docs_list"]
Path_docs_path = params["edinet"]["paths"]["docs_path"]

### 提出書類一覧を取得

In [29]:
today = date.today()

In [53]:
### 提出書類一覧を取得
def get_docs_list(date):
    params = {
        "date": date,
        "type": "2",
        "Subscription-Key": EDINET_API_KEY
    }
    
    res = requests.get(edinet_base_url + Path_docs_list, params=params)
    
    if res.status_code == 200:
        d = res.json()
        extracted_data = d["results"]
    
    elif res.status_code == 429:
        wait = int(res.headers.get("Retry-After", 60))
        print(f"Rate limit. Sleeping {wait} seconds...")
        time.sleep(wait)
    
    else:
        print(res.status_code)
        print(res.json())

    return extracted_data

# 1年前から今日までの日付のリストを作成
date_list = [
    (today - timedelta(days=i)).strftime(format="%Y-%m-%d")
    for i in range(0, 400)
]

# 各日付で決算書類リストを取得（並列処理）
docs_list = Parallel(n_jobs=-1)(
    delayed(get_docs_list)(date) for date in date_list
)

# 取得したデータをデータフレーム化
docs_df = pd.concat(
    pd.DataFrame(docs)
    for docs in docs_list
)

docs_df

# docs_list = []
# for date in date_list[0:30]:
#     docs_list += get_docs_list(date)
# docs_list

,seqNumber,docID,edinetCode,secCode,JCN,filerName,fundCode,ordinanceCode,formCode,docTypeCode,...,opeDateTime,withdrawalStatus,docInfoEditStatus,disclosureStatus,xbrlFlag,pdfFlag,attachDocFlag,englishDocFlag,csvFlag,legalStatus
0,1,S100Y1TB,E26349,NaN,4010401097969,マイルストーン・キャピタル・マネジメント株式会社,NaN,060,010002,350,...,NaN,0,0,0,1,1,0,0,1,1
1,2,S100Y17F,E12444,NaN,8010001114914,三井住友トラスト・アセットマネジメント株式会社,G15899,030,04A000,030,...,NaN,0,0,0,1,1,1,0,1,1
2,3,S100XZQA,E12425,NaN,5010001048652,フランクリン・テンプルトン・ジャパン株式会社,G10790,030,995000,180,...,NaN,0,0,0,1,1,0,0,1,1
3,4,S100XWAW,E12460,NaN,7010001054021,野村アセットマネジメント株式会社,G02839,030,04A000,030,...,NaN,0,0,0,1,1,1,0,1,1
4,5,S100XZX5,E40574,NaN,NaN,薄田 章博,NaN,060,010002,350,...,NaN,0,0,0,1,1,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
778,779,S100VJ9E,E03695,85180,6010001032820,日本アジア投資株式会社,NaN,010,053000,180,...,None,0,0,0,1,1,0,0,1,1
779,780,S100VJFP,E04358,96030,6011101002696,株式会社エイチ・アイ・エス,NaN,010,053000,180,...,None,0,0,0,1,1,0,0,1,1
780,781,S100VJFO,E01901,68560,1130001011676,株式会社堀場製作所,NaN,010,053000,180,...,None,0,0,0,1,1,0,0,1,1
781,782,S100VJEC,E23303,NaN,NaN,スーパーファンド・ジャパン・トレーディング（ケイマン）リミテッド,G07555,030,04C001,040,...,None,0,0,0,0,1,0,0,0,1


In [ ]:
Path('../output/edinet/document_list').mkdir(parents=True, exist_ok=True)
docs_df.to_parquet(f"../output/edinet/document_list/document_list.parquet")

In [55]:
docs_df = pd.read_parquet(f"../output/edinet/document_list/document_list.parquet")

In [109]:
# 銘柄コードがある行のみに絞り込み
docs_df = docs_df[docs_df["secCode"].notna()].reset_index(drop=True)

# 有価証券報告書のみに絞り込み
docs_df = docs_df[(docs_df["ordinanceCode"] == "010") & (docs_df["formCode"] == "030000")].reset_index(drop=True)

# 各銘柄コード最新の日付のみに絞る
docs_df = docs_df.loc[docs_df.groupby("secCode")["submitDateTime"].idxmax()]

# 書類IDと銘柄コードのみをリストで抽出
processed_docs_list = list(docs_df[["docID", "secCode"]].itertuples(index=False, name=None))
processed_docs_list

[('S100W543', '13010'),
 ('S100XSIT', '130A0'),
 ('S100W2X5', '13320'),
 ('S100W172', '13330'),
 ('S100VS6D', '135A0'),
 ('S100W4PR', '13750'),
 ('S100WL4U', '13760'),
 ('S100WKSN', '13770'),
 ('S100W0ZY', '13790'),
 ('S100WKSO', '137A0'),
 ('S100W98M', '13800'),
 ('S100WPSG', '13810'),
 ('S100WPAF', '13820'),
 ('S100XHX5', '13830'),
 ('S100W83V', '13840'),
 ('S100XMX3', '138A0'),
 ('S100WLH6', '14010'),
 ('S100X68L', '14070'),
 ('S100WQS8', '14140'),
 ('S100VZQA', '14170'),
 ('S100VTVY', '14180'),
 ('S100WL4I', '14190'),
 ('S100WRQT', '141A0'),
 ('S100W59G', '14200'),
 ('S100XUQB', '14290'),
 ('S100W7ZL', '142A0'),
 ('S100WLC9', '14300'),
 ('S100WQNU', '14310'),
 ('S100XZ7G', '14330'),
 ('S100X6GM', '14340'),
 ('S100XSYD', '14350'),
 ('S100WEBB', '14360'),
 ('S100XCY4', '14380'),
 ('S100W1I6', '143A0'),
 ('S100W3WQ', '14430'),
 ('S100WX6F', '14440'),
 ('S100XBZH', '14460'),
 ('S100W8I9', '14470'),
 ('S100XT84', '14490'),
 ('S100W14I', '14500'),
 ('S100XRTM', '145A0'),
 ('S100XUIC', '1

### 決算関連の書類を特定

In [111]:
def save_securities_report(docID, secCode):
    url = edinet_base_url + Path_docs_path

    url = url.replace("{docID}", docID)
    url = url.replace("{EDINET_API_KEY}", EDINET_API_KEY)

    try:
        # ファイルの読み込み
        res = requests.get(url)
        res.raise_for_status()
        
        content = res.content              
        
        # ファイルの保存
        output_path = f'../output/edinet/securities_report/{secCode}.zip'
        with open(output_path, 'wb') as file_out:
            file_out.write(content)

    except requests.exceptions.HTTPError as e:
        sys.stderr.write(str(e) + '\n')

Path('../output/edinet/securities_report').mkdir(parents=True, exist_ok=True)

Parallel(n_jobs=-1)(
    delayed(save_securities_report)(docID, secCode) for docID, secCode in processed_docs_list
)

# for docID, secCode in processed_docs_list[0:10]:
#     print(docID, secCode)
    
#     url = edinet_base_url + Path_docs_path

#     url = url.replace("{docID}", docID)
#     url = url.replace("{EDINET_API_KEY}", EDINET_API_KEY)

#     try:
#         # ファイルの読み込み
#         res = requests.get(url)
#         res.raise_for_status()
        
#         content = res.content              
        
#         # ファイルの保存
#         output_path = f'../output/edinet/securities_report/{secCode}.zip'
#         with open(output_path, 'wb') as file_out:
#             file_out.write(content)

#     except requests.exceptions.HTTPError as e:
#         sys.stderr.write(str(e) + '\n')

KeyboardInterrupt: 

### 決算関連の書類をダウンロード

In [ ]:
zip_path = "../output/edinet/securities_report/130A0.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open('XBRL/PublicDoc/xxx.xbrl') as f:
        content = f.read()